## Table of contents 
1.Imports & Dependencies\
2.Load & Preprocess Data\
3.Feature Engineering\
4.Oversampling Methods for Imbalanced Data\
5.Train & Evaluate Models\
6.Train & Evaluate Isolation Forest for Anomaly Detection

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/creditcardfraud/creditcard.csv


## 1.Imports & Dependencies
Load necessary libraries (Pandas, NumPy, TensorFlow, Sklearn, Flask, Imbalanced-Learn).


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, SVMSMOTE, ADASYN
from flask import Flask, request, jsonify
from collections.abc import Sequence

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/creditcardfraud


In [4]:
import os
# List downloaded files
files = os.listdir(path)
print(files) 

['creditcard.csv']


## 2.Load & Preprocess Data
1.Load creditcard.csv.\
2.Standardize Amount column.\
3.Drop Time column.


In [5]:
# Load the dataset
df = pd.read_csv(os.path.join(path, 'creditcard.csv'))
print(df.head())  # Display first 5 rows

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

## 3.Feature Engineering
1.Remove highly correlated features (above 0.95).\
2.Extract transaction patterns\
3.Transaction_Count_1hr: Count of transactions in the last hour.\
4.Avg_Amount_1hr: Average transaction amount in the last hour.\
5.Time_Since_Last_Transaction: Time difference between consecutive transactions.

In [6]:
# Preprocess Data
scaler = StandardScaler()
df["Amount"] = scaler.fit_transform(df["Amount"].values.reshape(-1, 1))
#df.drop(["Time"], axis=1, inplace=True)

In [7]:
# Convert "Time" to datetime format before dropping it
df["Time"] = pd.to_datetime(df["Time"], unit="s")  # Convert time to datetime

# Compute time-based transaction patterns before dropping "Time"
df['Time_Since_Last_Transaction'] = df["Time"].diff().dt.total_seconds().fillna(0)

# Now it's safe to drop "Time" column
df.drop(["Time"], axis=1, inplace=True)

# Extract Transaction Patterns
df['Transaction_Count_1hr'] = df['Class'].rolling(window=60).sum().fillna(0)  # Transactions in the last hour
df['Avg_Amount_1hr'] = df['Amount'].rolling(window=60).mean().fillna(0)  # Avg amount in last hour

In [8]:
print("Checking for NaN values before correlation computation:")
print(df.isnull().sum())

Checking for NaN values before correlation computation:
V1                             0
V2                             0
V3                             0
V4                             0
V5                             0
V6                             0
V7                             0
V8                             0
V9                             0
V10                            0
V11                            0
V12                            0
V13                            0
V14                            0
V15                            0
V16                            0
V17                            0
V18                            0
V19                            0
V20                            0
V21                            0
V22                            0
V23                            0
V24                            0
V25                            0
V26                            0
V27                            0
V28                            0
Amount              

In [9]:
df.fillna(0, inplace=True)  # Replaces NaNs with 0
# OR
df.dropna(inplace=True)  # Removes rows with NaNs

In [10]:
constant_columns = [col for col in df.columns if df[col].nunique() == 1]
print("Constant Columns:", constant_columns)

df.drop(columns=constant_columns, inplace=True)

Constant Columns: []


In [11]:
import pandas as pd
import numpy as np

def drop_highly_correlated_features(df, correlation_threshold=0.95):
    if df.empty:
        return df, []

    # Check for and drop columns with zero variance
    zero_variance_cols = df.columns[df.nunique() <= 1].tolist()
    if zero_variance_cols:
        print(f"Dropping columns with zero variance: {zero_variance_cols}")
        df = df.drop(columns=zero_variance_cols, errors='ignore')

    corr_matrix = df.corr()
    corr_matrix = corr_matrix.fillna(0)

    # Check for remaining NaNs
    if corr_matrix.isnull().values.any():
        print("WARNING: NaNs still present in correlation matrix after fillna!")
        nan_cols = corr_matrix.columns[corr_matrix.isnull().any()].tolist()
        print("Columns with NaNs:", nan_cols)

    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(abs(upper[column]) > correlation_threshold)]
    df = df.drop(columns=to_drop, errors='ignore')

    return df, to_drop

In [12]:
def extract_transaction_patterns(df):
    df['Time_Since_Last_Transaction'] = df.index.to_series().diff().fillna(0)
    return df

## 4.Oversampling Methods for Imbalanced Data
1.SMOTE (Synthetic Minority Over-sampling Technique)\
2.Borderline-SMOTE\
3.SVM-SMOTE\
4.ADASYN (Adaptive Synthetic Sampling)\
5.Print class distributions after oversampling.

In [13]:
# Oversampling Methods
X = df.drop(columns=["Class"])
y = df["Class"]

In [14]:
smote = SMOTE(sampling_strategy=0.2, random_state=42)
borderline_smote = BorderlineSMOTE(sampling_strategy=0.2, random_state=42)
svm_smote = SVMSMOTE(sampling_strategy=0.2, random_state=42)
adasyn = ADASYN(sampling_strategy=0.2, random_state=42)

X_smote, y_smote = smote.fit_resample(X, y)
X_borderline, y_borderline = borderline_smote.fit_resample(X, y)
X_svm, y_svm = svm_smote.fit_resample(X, y)
X_adasyn, y_adasyn = adasyn.fit_resample(X, y)

print("Class Distribution after SMOTE:", np.bincount(y_smote))
print("Class Distribution after Borderline-SMOTE:", np.bincount(y_borderline))
print("Class Distribution after SVM-SMOTE:", np.bincount(y_svm))
print("Class Distribution after ADASYN:", np.bincount(y_adasyn))

Class Distribution after SMOTE: [284315  56863]
Class Distribution after Borderline-SMOTE: [284315  56863]
Class Distribution after SVM-SMOTE: [284315  56863]
Class Distribution after ADASYN: [284315  56861]


## 5.Train & Evaluate Models
1.Define evaluate_model function.\
2.Train/test split for each oversampling method.\
3.Train Random Forest with:\
    A.SMOTE\
    B.Borderline-SMOTE\
    C.SVM-SMOTE\
    D.ADASYN\
    E.Print classification reports for each.

In [15]:
# Evaluate Models after Different Oversampling Methods
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name} Model Performance:")
    print(classification_report(y_test, y_pred))

In [16]:
from sklearn.model_selection import train_test_split

X_train_smote, X_test_smote, y_train_smote, y_test_smote = train_test_split(X_smote, y_smote, test_size=0.2, random_state=42)
X_train_borderline, X_test_borderline, y_train_borderline, y_test_borderline = train_test_split(X_borderline, y_borderline, test_size=0.2, random_state=42)
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(X_svm, y_svm, test_size=0.2, random_state=42)
X_train_adasyn, X_test_adasyn, y_train_adasyn, y_test_adasyn = train_test_split(X_adasyn, y_adasyn, test_size=0.2, random_state=42)

In [17]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
evaluate_model(rf, X_train_smote, y_train_smote, X_test_smote, y_test_smote, "Random Forest (SMOTE)")
evaluate_model(rf, X_train_borderline, y_train_borderline, X_test_borderline, y_test_borderline, "Random Forest (Borderline-SMOTE)")
evaluate_model(rf, X_train_svm, y_train_svm, X_test_svm, y_test_svm, "Random Forest (SVM-SMOTE)")
evaluate_model(rf, X_train_adasyn, y_train_adasyn, X_test_adasyn, y_test_adasyn, "Random Forest (ADASYN)")


Random Forest (SMOTE) Model Performance:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56966
           1       1.00      1.00      1.00     11270

    accuracy                           1.00     68236
   macro avg       1.00      1.00      1.00     68236
weighted avg       1.00      1.00      1.00     68236

Random Forest (Borderline-SMOTE) Model Performance:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56966
           1       1.00      1.00      1.00     11270

    accuracy                           1.00     68236
   macro avg       1.00      1.00      1.00     68236
weighted avg       1.00      1.00      1.00     68236

Random Forest (SVM-SMOTE) Model Performance:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56966
           1       1.00      1.00      1.00     11270

    accuracy                           1.00 

The results show perfect classification performance (precision, recall, and F1-score of 1.00) for all the Random Forest models trained with different oversampling techniques (SMOTE, Borderline-SMOTE, SVM-SMOTE, and ADASYN). This is a strong red flag and suggests a few potential issues:

Perfect Scores:
Achieving 1.00 for precision, recall, and F1-score across all classes and for all metrics is extremely rare in real-world scenarios. It indicates that the model is perfectly predicting every instance, which is highly suspicious.

## 6.Train & Evaluate Isolation Forest for Anomaly Detection
1.Train Isolation Forest with contamination rate of 0.02.\
2.Predict anomalies and map {1: 0, -1: 1} (fraud label).\
3.Print classification report.

In [18]:
# Train Isolation Forest model
iso_forest = IsolationForest(n_estimators=100, contamination=0.02)
df["Iso_Anomaly"] = iso_forest.fit_predict(df.drop("Class", axis=1))
df["Iso_Anomaly"] = df["Iso_Anomaly"].map({1: 0, -1: 1})
print("Isolation Forest Model Performance:")
print(classification_report(df["Class"], df["Iso_Anomaly"]))

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Isolation Forest Model Performance:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    284315
           1       0.07      0.82      0.13       492

    accuracy                           0.98    284807
   macro avg       0.54      0.90      0.56    284807
weighted avg       1.00      0.98      0.99    284807



Let's break down the Isolation Forest model performance results:

**Understanding the Metrics:**

* **Precision:**
    * For class 0 (likely the majority class, "normal" or non-anomalous), precision is 1.00, meaning when the model predicts class 0, it's always correct.
    * For class 1 (likely the minority class, "anomalous"), precision is 0.06, meaning only 6% of the instances predicted as class 1 are actually class 1. This is very poor.
* **Recall:**
    * For class 0, recall is 0.98, meaning the model correctly identifies 98% of all actual class 0 instances.
    * For class 1, recall is 0.74, meaning the model identifies 74% of all actual class 1 instances. This is relatively better than the precision, but still indicates that around 26% of the anomalies are missed.
* **F1-score:**
    * The F1-score is the harmonic mean of precision and recall.
    * For class 0, it's 0.99, which is excellent.
    * For class 1, it's 0.12, which is very low, reflecting the poor precision.
* **Support:**
    * Support indicates the number of actual occurrences of each class.
    * Class 0 has 284,315 instances, and class 1 has 492 instances, clearly showing a significant class imbalance.
* **Accuracy:**
    * The overall accuracy is 0.98, which looks good at first glance. However, accuracy can be misleading in imbalanced datasets, as it's heavily influenced by the majority class.
* **Macro Avg:**
    * The macro average calculates the average of precision, recall, and F1-score without considering class imbalance.
* **Weighted Avg:**
    * The weighted average accounts for class imbalance by weighting the metrics based on the number of instances in each class.

**Analysis:**

* **Class Imbalance:**
    * The enormous difference in support between class 0 and class 1 is the most critical factor. The dataset is heavily imbalanced.
* **Good Performance on Majority Class:**
    * The model performs very well on the majority class (0), as evidenced by the high precision, recall, and F1-score.
* **Poor Performance on Minority Class:**
    * The model struggles significantly with the minority class (1), resulting in very low precision and a low F1 score. The model is labeling many normal cases as anomalies.
* **High Recall, Low Precision for Anomalies:**
    * The model is finding a large percentage of the anomalies, but it is also flagging a very large amount of normal data as anomalies.
* **Accuracy Misleading:**
    * The high accuracy (0.98) is misleading because it's driven by the model's ability to correctly classify the majority class.
* **Isolation Forest Characteristics:**
    * Isolation Forest is designed for anomaly detection. It tends to work well for detecting anomalies that are very different from the normal data. However, it can struggle when anomalies are subtle or when there's significant overlap between normal and anomalous data.

**Key Takeaways and Recommendations:**

* **Focus on Precision and Recall for Anomalies:**
    * In anomaly detection, especially with imbalanced data, focus on the precision and recall of the minority class (anomalies).
* **Consider Alternative Metrics:**
    * Metrics like precision-recall curves (PR curves) and area under the PR curve (AUC-PR) are more informative in imbalanced datasets.
* **Adjust Thresholds:**
    * Isolation Forest assigns anomaly scores. You can adjust the threshold for classifying instances as anomalies to improve precision or recall, depending on your priorities.
* **Feature Engineering:**
    * Explore feature engineering to create features that better distinguish between normal and anomalous instances.
* **Algorithm Selection:**
    * Consider other anomaly detection algorithms that might be more suitable for your data, such as One-Class SVM or Local Outlier Factor (LOF).
* **Collect more anomaly data:**
    * If possible, collecting more anomaly data would help the model learn the characteristics of the anomolous class better.
* **Resampling Techniques:**
    * While Isolation Forest doesn't directly use oversampling or undersampling during training, you could explore using them to create a more balanced dataset for feature analysis or for training a different classification model if needed.


Thanks!